# 0 - Importação das bibliotecas utilizadas no projeto

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import re

# 1 - Inicio importação dos datasets do modelo

In [ ]:
# Importação dos datasets

print("====== LABEL ======")
df_tc_labels = pd.read_csv('../src/data/raw/medical_tc_labels.csv')
df_tc_labels.info()

print("====== TEST ======")
df_tc_test = pd.read_csv('../src/data/raw/medical_tc_test.csv')
df_tc_test.info()


print("====== TRAIN ======")
df_tc_train = pd.read_csv('../src/data/raw/medical_tc_train.csv')
df_tc_train.info()

# 2 - Analise dos dados dos datasets

In [ ]:
# Visualização dos dados em gráficos

x = df_tc_test.groupby('condition_label')['medical_abstract'].count()
print(x)

In [ ]:
# Visualização dos dados em gráficos

x.plot(kind='bar', color='skyblue', edgecolor='black')

plt.title("Total de medical_abstract por condition_label")
plt.xlabel("condition label")
plt.ylabel("total medical abstract")
plt.tight_layout
plt.show()

In [ ]:
# Visualização dos dados em gráficos

sns.countplot(data=df_tc_test, x='condition_label',palette='pastel')
plt.title("Total de medical_abstract por condition_label")
plt.show()

In [ ]:
# Visualização dos dados em gráficos

sns.countplot(data=df_tc_train, x='condition_label',palette='pastel')
plt.title("Total de medical_abstract por condition_label")
plt.show()

# 3 - Construção do dataset que será utilizado para tratamento e treinamento do modelo

In [12]:
# Cruzamento para capturar a informação de classificação conforme a condição

df_train_completo = df_tc_train.merge(df_tc_labels, how='inner', on='condition_label')
df_train_completo.to_csv('../src/data/processed/df_train_completo.csv')

df_test_completo = df_tc_test.merge(df_tc_labels, how='inner', on='condition_label')
df_test_completo.to_csv('../src/data/processed/df_test_completo.csv')

# 4 - Tratamento de dados para o modelo

In [13]:
import spacy
nlp = spacy.load("en_core_web_sm")
print(nlp)

In [14]:
documento = nlp("im learning english")
documento

im learning english

In [15]:
for token in documento:
    print(token.text, token.pos_)

i PRON
m AUX
learning VERB
english PROPN


Lematização: lema de uma palavra de acordo com seu significado no dicionarios
Stemização: extrair o radical das palavras

In [16]:
documento_df = nlp(str(df_train_completo['medical_abstract']))

for token in documento_df:
    print(token.text, token.pos_)

0 NUM
        SPACE
Tissue PROPN
changes NOUN
around ADP
loose ADJ
prostheses NOUN
. PUNCT
A DET
cani NOUN
... PUNCT

 SPACE
1 NUM
        SPACE
Neuropeptide PROPN
Y PROPN
and CCONJ
neuron NOUN
- PUNCT
specific ADJ
enolase NOUN
lev NOUN
... PUNCT

 SPACE
2 NUM
        SPACE
Sexually ADV
transmitted VERB
diseases NOUN
of ADP
the DET
colon NOUN
, PUNCT
re NOUN
... PRON

 SPACE
3 NUM
        SPACE
Lipolytic ADJ
factors NOUN
associated VERB
with ADP
murine NOUN
and CCONJ
h PROPN
... PUNCT

 SPACE
4 NUM
        SPACE
Does AUX
carotid NOUN
restenosis NOUN
predict VERB
an DET
increased VERB
r NOUN
... PUNCT

                                SPACE
... PUNCT
                       
 SPACE
11545 NUM
    SPACE
Epirubicin PROPN
at ADP
two NUM
dose NOUN
levels NOUN
with ADP
prednisolon NOUN
... PUNCT

 SPACE
11546 NUM
    SPACE
Four NUM
and CCONJ
a DET
half NOUN
year NOUN
follow VERB
up ADP
of ADP
women NOUN
with ADP
d PROPN
... PUNCT

 SPACE
11547 NUM
    SPACE
Safety PROPN
of ADP
the DET
transbron

In [ ]:
for entidade in documento_df.ents:
    print(entidade.text, entidade.label_)

In [ ]:
from spacy import displacy
displacy.render(documento_df, style='ent', jupyter=True)

In [ ]:
import nltk
nltk.download('rslp')

In [ ]:
stemmet = nltk.stem.RSLPStemmer()
stemmet.stem('aprendendo')

In [ ]:
for token in documento_df:
    print(token.text, token.lemma, stemmet.stem(token.text))

In [ ]:
# stopwords = palavras que nao agregam
from spacy.lang.en.stop_words import STOP_WORDS
len(STOP_WORDS)

In [ ]:
nlp.vocab['would'].is_stop

In [ ]:
# Funções de pre processamento: retirada e pontuações, remocao de stop words
from spacy.lang.en.stop_words import STOP_WORDS
import string

stop_words = STOP_WORDS
print(stop_words)

pontuacoes = string.punctuation
print(pontuacoes)

In [ ]:
pln = spacy.load("en_core_web_sm")
pln

In [ ]:
def pre_processamento(texto):
    texto = texto.lower()
    documento = pln(texto)

    lista = []
    for token in documento:
        lista.append(token.lemma_)

    lista = [palavra for palavra in lista if palavra not in stop_words and palavra not in pontuacoes]
    lista = ' '.join([str(elemento) for elemento in lista if not elemento.isdigit()])
    return lista

In [ ]:
teste = pre_processamento('Im learning, @ the ENGLISH ; ')

print(teste)

In [18]:
df_train_completo.head(2)

,condition_label,medical_abstract,condition_name
0,5,Tissue changes around loose prostheses. A cani...,general pathological conditions
1,1,Neuropeptide Y and neuron-specific enolase lev...,neoplasms


In [19]:
# Função para normalizar texto
def lower_replace(text: str) -> str:
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)       # remove conteúdo entre colchetes
    text = re.sub(r'[^\w\s]', '', text)       # remove pontuação
    return text

# Tokenização + lematização + remoção de stopwords
def token_lemma_stop(text: str) -> list:
    doc = nlp(text)
    return [token.lemma_ for token in doc if not token.is_stop]

# Filtrar apenas certas classes gramaticais (exemplo: substantivos e adjetivos)
def filter_pos(tokens: list) -> str:
    doc = nlp(" ".join(tokens))
    return " ".join([token.text for token in doc if token.pos_ in ["NOUN", "ADJ", "PRON", "VERB"]])


# Pipeline único
def preprocess(text: str) -> list:
    text = lower_replace(text)
    tokens = token_lemma_stop(text)
    return filter_pos(tokens)

# Aplicar no DataFrame
df_train_completo['medical_abstract_clean'] = df_train_completo['medical_abstract'].apply(preprocess)


In [20]:
df_train_completo

,condition_label,medical_abstract,condition_name,medical_abstract_clean
0,5,Tissue changes around loose prostheses. A cani...,general pathological conditions,tissue change loose prosthesis canine model in...
1,1,Neuropeptide Y and neuron-specific enolase lev...,neoplasms,neuronspecific enolase level benign malignant ...
2,2,"Sexually transmitted diseases of the colon, re...",digestive system diseases,transmit disease colon rectum anus challenge p...
3,1,Lipolytic factors associated with murine and h...,neoplasms,lipolytic factor associate human cancer identi...
4,3,Does carotid restenosis predict an increased r...,nervous system diseases,carotid restenosis predict increase risk late ...
...,...,...,...,...
11545,1,Epirubicin at two dose levels with prednisolon...,neoplasms,level prednisolone treatment advanced breast c...
11546,1,Four and a half year follow up of women with d...,neoplasms,half year follow woman dyskaryotic cervical sm...
11547,5,Safety of the transbronchial biopsy in outpati...,general pathological conditions,safety transbronchial biopsy outpatient object...
11548,3,Interictal spikes and hippocampal somatostatin...,nervous system diseases,somatostatin level temporal lobe epilepsy inve...


In [23]:
df_train_completo.to_csv('../src/data/processed/df_train_processed.csv')

In [26]:
df_train_preprocessed = pd.read_csv('../src/data/processed/df_train_processed.csv')
df_train_preprocessed.head(10)

,Unnamed: 0,condition_label,medical_abstract,condition_name,medical_abstract_clean
0,0,5,Tissue changes around loose prostheses. A cani...,general pathological conditions,tissue change loose prosthesis canine model in...
1,1,1,Neuropeptide Y and neuron-specific enolase lev...,neoplasms,neuronspecific enolase level benign malignant ...
2,2,2,"Sexually transmitted diseases of the colon, re...",digestive system diseases,transmit disease colon rectum anus challenge p...
3,3,1,Lipolytic factors associated with murine and h...,neoplasms,lipolytic factor associate human cancer identi...
4,4,3,Does carotid restenosis predict an increased r...,nervous system diseases,carotid restenosis predict increase risk late ...
5,5,3,The shoulder in multiple epiphyseal dysplasia....,nervous system diseases,multiple epiphyseal dysplasia shoulder assess ...
6,6,2,The management of postoperative chylous ascite...,digestive system diseases,management postoperative chylous ascite case r...
7,7,4,Pharmacomechanical thrombolysis and angioplast...,cardiovascular diseases,pharmacomechanical thrombolysis angioplasty ma...
8,8,5,Color Doppler diagnosis of mechanical prosthet...,general pathological conditions,color doppler diagnosis mechanical prosthetic ...
9,9,5,Noninvasive diagnosis of right-sided extracard...,general pathological conditions,noninvasive diagnosis rightside extracardiac c...


In [27]:
df_train_final = []

for medical_abstract, condition_name in zip(df_train_preprocessed['medical_abstract'], df_train_preprocessed['condition_name']):
    if condition_name == 'neoplasms':
        dic = ({'neoplasms': True, 'digestive system diseases': False, 'nervous system diseases':False, 'cardiovascular diseases': False, 'general pathological conditions': False})
    elif condition_name == 'digestive system diseases':
        dic = ({'neoplasms': False, 'digestive system diseases': True, 'nervous system diseases':False, 'cardiovascular diseases': False, 'general pathological conditions': False})   
    elif condition_name == 'nervous system diseases':
            dic = ({'neoplasms': False, 'digestive system diseases': False, 'nervous system diseases':True, 'cardiovascular diseases': False, 'general pathological conditions': False})
    elif condition_name == 'cardiovascular diseases':
                dic = ({'neoplasms': False, 'digestive system diseases': False, 'nervous system diseases':False, 'cardiovascular diseases': True, 'general pathological conditions': False})
    elif condition_name == 'general pathological conditions':
                dic = ({'neoplasms': False, 'digestive system diseases': False, 'nervous system diseases':False, 'cardiovascular diseases': False, 'general pathological conditions': True})     

    df_train_final.append([medical_abstract, dic.copy()])             

In [28]:
df_train_final[0][1]

{'neoplasms': False,
 'digestive system diseases': False,
 'nervous system diseases': False,
 'cardiovascular diseases': False,
 'general pathological conditions': True}

In [29]:
type(df_train_final[0][1])

dict

In [30]:
import random
import spacy
from spacy.training.example import Example
from spacy.util import compounding

# Criar modelo em branco
modelo = spacy.blank("en")

# Adicionar componente de classificação de texto
categorias = modelo.add_pipe("textcat", last=True)
categorias.add_label("neoplasms")
categorias.add_label("digestive system diseases")
categorias.add_label("nervous system diseases")
categorias.add_label("cardiovascular diseases")
categorias.add_label("general pathological conditions")

# Inicializar otimizador DEPOIS de adicionar o textcat
optimizer = modelo.initialize()

historico = []

batch_sizes = compounding(32.0, 512.0, 1.001)

for epoca in range(10):
    random.shuffle(df_train_final)
    losses = {}

    # Criar batches
    for batch in spacy.util.minibatch(df_train_final, size=batch_sizes):
        examples = []
        for texto, entities in batch:
            doc = modelo.make_doc(texto)
            examples.append(Example.from_dict(doc, {"cats": entities}))

        # Atualizar o modelo com os exemplos
        modelo.update(examples, sgd=optimizer, losses=losses)

    # Mostrar perdas acumuladas da época
    if epoca % 5 == 0:
        print(f"Época {epoca} - Losses: {losses}")
        historico.append(losses.copy())


Época 0 - Losses: {'textcat': 38.04792424291372}
Época 5 - Losses: {'textcat': 8.28568671271205}


In [ ]:
modelo.to_disk('modelo')

In [ ]:
historico_loss = []
for i in historico:
    historico_loss.append(i.get('textcat'))


historico_loss = np.array(historico_loss)
historico_loss

plt.plot(historico_loss)
plt.title('progressao do erro')
plt.xlabel('epocas')
plt.ylabel('erro')

In [ ]:
modelo_carregado = spacy.load("modelo") 
modelo_carregado


frase = 'I have a headache.'

frase = pre_processamento(frase)
frase

In [ ]:
previsao = modelo_carregado(frase)
previsao
previsao.cats